# Python Standard Library & Useful Modules Bootcamp

A practical, detailed guide to 12 Python modules and libraries:
`subprocess`, `requests`, `sys`, `os`, `collections`, `random`, `re`, `string`, `pathlib`, `shutil`, `urllib`, and `asyncio`.

## Structure
- 12 modules, each divided into four focused lessons (48 lessons total).
- Every lesson uses two cells: a Markdown explanation, followed by commented Python code.
- Examples use temporary folders for file operations and mocked network responses where possible.
- Examples aim to run without external files or network access; the `requests` module must be installed.

## Setup
Most modules here are in Python's standard library. `requests` is third-party; install it if necessary:

```python
%pip install requests
```

Run this notebook from top to bottom or run individual code cells. The `asyncio` lessons use `await` directly, which is suitable for Jupyter notebooks; in a standalone Python script, use `asyncio.run()` at the program entry point.

## Safety and good practice
- Use `subprocess` with argument lists; avoid `shell=True` with untrusted input.
- Never print or commit API keys and tokens.
- Use timeouts for network and child-process operations.
- Only access websites and endpoints you are authorized to use; respect site policies.
- Use `TemporaryDirectory` for examples that create or delete folders.
- `random` is for simulation, not security; use `secrets` for security-sensitive random values.


# 1. subprocess

## Run a command safely with `subprocess.run`

`subprocess` lets Python start and control external programs, such as Git, another Python script, or a system utility.

For most tasks, start with `subprocess.run()`. Pass the program and its arguments as a **list** instead of building a shell command string. This avoids many quoting problems and reduces shell-injection risk. `capture_output=True` captures standard output and standard error; `text=True` makes the captured values strings rather than bytes.

The example launches the current Python interpreter and asks the child process to print a message. This is portable and does not depend on whether a shell command such as `ls` or `dir` exists.

In [ ]:
import subprocess  # Import tools for starting external processes.
import sys  # Find the Python executable running this notebook.

command = [sys.executable, "-c", "print('Hello from a child process')"]  # Build an argument list for a tiny Python program.
result = subprocess.run(command, capture_output=True, text=True, check=True)  # Run it, capture output, and raise an error if it fails.
print("Return code:", result.returncode)  # A return code of zero usually indicates success.
print("Standard output:", result.stdout.strip())  # Read the child's normal output.
print("Standard error:", result.stderr.strip() or "(empty)")  # Display errors if the child wrote any.

## Pass arguments and input without using a shell

Each item in the command list is one argument. Python handles spaces and quoting in arguments without needing you to manually escape them for a shell.

Use `input=` when the child program should receive text on standard input. Avoid `shell=True` for values influenced by users or external data: shell metacharacters can change what gets executed. If you genuinely need shell syntax, separate it from untrusted data and understand the security consequences.

The child program below receives a phrase with spaces as one argument and receives a second message through standard input.

In [ ]:
import subprocess  # Import subprocess functionality.
import sys  # Access the active Python interpreter.

child_code = "import sys; print('ARG:', sys.argv[1]); print('INPUT:', sys.stdin.read().strip())"  # Define a small child program.
command = [sys.executable, "-c", child_code, "phrase with spaces"]  # Keep the phrase as a single argument.
result = subprocess.run(command, input="Message from standard input\n", capture_output=True, text=True, check=True)  # Send input and collect output.
print(result.stdout)  # Display both values printed by the child.

## Handle errors, return codes, and timeouts

By default, `run()` returns a `CompletedProcess`, even if the child exits unsuccessfully. `check=True` converts a non-zero return code into `subprocess.CalledProcessError`, which is useful when failure should stop the workflow.

Use `timeout=` when a child process must not run indefinitely. If the time limit is exceeded, Python raises `subprocess.TimeoutExpired`. You can inspect `returncode`, `stdout`, and `stderr` to diagnose a failed command. Do not catch every exception indiscriminately; handle the failures you can recover from.

In [ ]:
import subprocess  # Import process control tools.
import sys  # Locate Python for portable child-process examples.

failed_code = "import sys; print('diagnostic message', file=sys.stderr); sys.exit(2)"  # Make a child process fail intentionally.
try:  # Start a block that may raise CalledProcessError.
    subprocess.run([sys.executable, "-c", failed_code], check=True, capture_output=True, text=True, timeout=5)  # Require success and impose a time limit.
except subprocess.CalledProcessError as error:  # Handle a completed command with a non-zero exit code.
    print("Failed return code:", error.returncode)  # Display the child's exit status.
    print("Error output:", error.stderr.strip())  # Display the child's error message.

try:  # Demonstrate the timeout boundary.
    subprocess.run([sys.executable, "-c", "import time; time.sleep(2)"], timeout=0.1, check=True)  # Stop waiting if the child exceeds the limit.
except subprocess.TimeoutExpired:  # Handle a command that takes too long.
    print("The child process exceeded the timeout.")  # Report the timeout clearly.

## Use `Popen` when you need more control

`subprocess.run()` is the simpler choice for a command that should finish before Python continues. `subprocess.Popen()` exposes the process while it is running, enabling patterns such as streaming output, connecting multiple processes, or polling status.

When using pipes with `Popen`, remember to read them and wait for the process. A child can block if a pipe fills and the parent does not consume it. For simple use cases, `communicate()` collects output and waits for completion. `Popen` is more flexible, but that flexibility comes with more responsibility.

In [ ]:
import subprocess  # Import the process API.
import sys  # Use the current Python interpreter.

command = [sys.executable, "-c", "print('first'); print('second')"]  # Define a short process that writes two lines.
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) as process:  # Start it and capture both streams.
    stdout, stderr = process.communicate(timeout=5)  # Read the streams and wait for the child to finish.
    return_code = process.returncode  # Read the process exit status after communicate().
print("Output lines:", stdout.splitlines())  # Display the captured lines.
print("Return code:", return_code)  # Confirm whether the child completed successfully.
print("Standard error:", stderr.strip() or "(empty)")  # Display any captured error output.

# 2. requests

## Make a GET request and inspect the response

The third-party `requests` library provides a convenient interface for HTTP. A response includes a status code, headers, and body. A successful request does not always mean the server returned a successful HTTP status, so call `raise_for_status()` when 4xx and 5xx responses should be treated as failures.

Most calls that access a real server should specify a timeout. Without one, a program can wait much longer than expected. This lesson mocks the HTTP response so it works without internet access; replace the mock with a permitted URL when testing your own integration.

In [ ]:
from unittest.mock import Mock, patch  # Provide a fake response so this example needs no network.
import requests  # Import the HTTP client library.

fake_response = Mock()  # Create a stand-in for requests.Response.
fake_response.status_code = 200  # Simulate a successful HTTP status.
fake_response.headers = {"Content-Type": "application/json"}  # Simulate response metadata.
fake_response.text = '{"status": "ok"}'  # Simulate the response body.
fake_response.json.return_value = {"status": "ok"}  # Define the parsed JSON payload.
with patch("requests.get", return_value=fake_response) as mocked_get:  # Replace requests.get only within this block.
    response = requests.get("https://api.example.test/status", timeout=5)  # Make a mocked request with a timeout.
    response.raise_for_status()  # Validate the HTTP status.
    print("Status:", response.status_code)  # Display the status code.
    print("Content type:", response.headers["Content-Type"])  # Display the content type.
    print("JSON:", response.json())  # Parse the JSON payload.
    print("Request called with:", mocked_get.call_args)  # Inspect the URL and timeout passed to the client.

## Send query parameters and headers

Use the `params` argument to build query strings and `headers` to supply request metadata. Let the library encode query parameters instead of manually concatenating strings; this handles spaces and special characters more reliably.

Do not put secrets such as API keys into source code or public notebooks. Use environment variables or a secrets manager, and avoid printing authorization headers into logs. The mocked call below demonstrates which values are sent without contacting an endpoint.

In [ ]:
from unittest.mock import Mock, patch  # Create a safe mock for the HTTP call.
import requests  # Import requests.

fake_response = Mock()  # Create a fake response object.
fake_response.status_code = 200  # Simulate HTTP success.
fake_response.json.return_value = {"items": [{"id": 1}, {"id": 2}]}  # Simulate a JSON API result.
params = {"q": "data analyst", "page": 2, "per_page": 10}  # Store query parameters as a dictionary.
headers = {"Accept": "application/json", "User-Agent": "LearningClient/1.0"}  # Define non-secret request headers.
with patch("requests.get", return_value=fake_response) as mocked_get:  # Intercept the request for this example.
    response = requests.get("https://api.example.test/search", params=params, headers=headers, timeout=5)  # Send parameters and headers to the mock.
    response.raise_for_status()  # Raise an exception if the HTTP status indicates failure.
    print("Payload:", response.json())  # Read the mocked API response.
    print("Called with:", mocked_get.call_args)  # Inspect the parameters and timeout.

## Post JSON and handle status errors

Use `json=payload` to send a Python dictionary as JSON. Requests serializes the object and sets an appropriate content type. Use `data=` for other body formats such as form-encoded data or already-encoded bytes.

A common robust pattern is to set a timeout, call `raise_for_status()`, and then parse the expected response. Catch network failures such as `requests.Timeout` and `requests.ConnectionError` where you can make a meaningful recovery decision. Do not assume every response body is JSON.

In [ ]:
from unittest.mock import Mock, patch  # Mock the POST request instead of sending data externally.
import requests  # Import the HTTP client.

payload = {"name": "sample-job", "active": True}  # Define a JSON payload for a hypothetical API.
fake_response = Mock()  # Build a mock response.
fake_response.status_code = 201  # Simulate successful resource creation.
fake_response.json.return_value = {"id": 42, **payload}  # Simulate the returned resource.
with patch("requests.post", return_value=fake_response) as mocked_post:  # Intercept the POST operation.
    response = requests.post("https://api.example.test/jobs", json=payload, timeout=5)  # Send JSON to the mock.
    response.raise_for_status()  # Validate the status code.
    print("Created resource:", response.json())  # Read the simulated created record.
    print("POST arguments:", mocked_post.call_args)  # Inspect the JSON body and timeout.

## Reuse connections with Session

`requests.Session` reuses connections and allows common headers, cookies, and settings to persist across requests. It is useful when making several calls to the same service.

Keep session lifetime explicit, usually by using `with requests.Session() as session:`. Apply timeouts to each request, and consider a deliberate retry strategy for idempotent operations. Do not retry every operation blindly: repeating a non-idempotent payment or create request could cause duplicate effects unless the API supports idempotency keys.

In [ ]:
from unittest.mock import Mock, patch  # Use a fake response to demonstrate a session call.
import requests  # Import requests.

fake_response = Mock()  # Make a response substitute.
fake_response.status_code = 200  # Simulate success.
fake_response.json.return_value = {"page": 1, "items": []}  # Provide a minimal response payload.
with requests.Session() as session:  # Create and automatically close a reusable session.
    session.headers.update({"Accept": "application/json", "User-Agent": "NotebookDemo/1.0"})  # Apply defaults to calls from this session.
    with patch.object(session, "get", return_value=fake_response) as mocked_get:  # Mock the session's GET method.
        response = session.get("https://api.example.test/items", params={"page": 1}, timeout=(3, 10))  # Set connect and read timeouts.
        response.raise_for_status()  # Validate status before using the body.
        print(response.json())  # Display the simulated payload.
        print("Request called:", mocked_get.call_args)  # Inspect the method arguments.

# 3. sys

## Inspect the Python runtime

The `sys` module exposes interpreter-related information and functions. Common values include `sys.version`, `sys.executable`, `sys.platform`, `sys.argv`, `sys.path`, `sys.modules`, `sys.stdin`, `sys.stdout`, and `sys.stderr`.

These are useful for diagnostics and command-line programs. `sys.executable` identifies the interpreter running the current process, which is often more reliable than hard-coding a path to `python`. Remember that platform strings differ across operating systems.

In [ ]:
import sys  # Import interpreter and process information.

print("Python version:", sys.version.split()[0])  # Display the version number.
print("Executable:", sys.executable)  # Display the Python executable path.
print("Platform:", sys.platform)  # Display the operating-system platform identifier.
print("Arguments:", sys.argv)  # Show command-line arguments passed to this process.
print("Module search paths:", len(sys.path))  # Count directories Python searches for imports.
print("Loaded modules:", len(sys.modules))  # Count modules currently loaded in this interpreter.

## Parse command-line arguments conceptually

`sys.argv` is a list of command-line arguments, where `sys.argv[0]` is normally the script name and later items are user-provided arguments. It is sufficient for very small scripts, but manual parsing becomes fragile as options grow.

For real command-line tools, `argparse` is generally preferred because it supports flags, types, defaults, help text, and validation. This function accepts an argument list so it can be tested without restarting the notebook.

In [ ]:
import argparse  # Import the standard command-line parser.
def parse_args(arguments):  # Accept an explicit list so the function is easy to test.
    parser = argparse.ArgumentParser(description="Summarize a CSV data file.")  # Define command purpose and help output.
    parser.add_argument("filename", help="Path to the input CSV file")  # Define one required positional argument.
    parser.add_argument("--limit", type=int, default=10, help="Maximum records to report")  # Define a typed optional argument.
    return parser.parse_args(arguments)  # Parse the provided list instead of reading the notebook's own arguments.

args = parse_args(["sales.csv", "--limit", "25"])  # Simulate the command: script.py sales.csv --limit 25.
print(args.filename)  # Display the selected input file.
print(args.limit)  # Display the parsed integer limit.

## Use standard streams and exit codes

Command-line programs normally write regular results to stdout and diagnostics to stderr. A process's exit code communicates success or failure to scripts and automation; by convention, zero means success and non-zero means a problem.

Instead of calling `sys.exit()` deep inside business logic, a maintainable pattern is for `main()` to return an integer status and for the script entry point to pass that status to `sys.exit()`. In a notebook, call `main()` directly so the kernel is not terminated.

In [ ]:
import sys  # Access standard output and error streams.
def main(value):  # Define a small command-style function.
    if value < 0:  # Validate the supplied input.
        print("Error: value must be non-negative", file=sys.stderr)  # Send diagnostics to stderr.
        return 2  # Return a non-zero status to signal failure.
    print(f"Value accepted: {value}", file=sys.stdout)  # Send normal output to stdout.
    return 0  # Return zero to signal success.

status = main(5)  # Run a successful example without exiting the notebook.
print("Exit status would be:", status)  # Show what the command-line entry point would return.

## Inspect imports and memory size

`sys.modules` is a dictionary of modules already loaded in the current interpreter. `sys.path` lists import search locations. These help diagnose environment issues, but modifying them casually can make imports confusing.

`sys.getsizeof()` reports the shallow size of one object, not the full memory footprint of all objects it references. It can be useful for comparisons, but should not be treated as a complete memory profiler.

In [ ]:
import sys  # Import interpreter diagnostics.

print("collections loaded:", "collections" in sys.modules)  # Check whether a standard library module is already loaded.
print("First import search path:", sys.path[0])  # Display the first search location, which can depend on how Python started.
sample = [1, 2, 3, 4, 5]  # Create a small object to inspect.
print("Shallow list size:", sys.getsizeof(sample), "bytes")  # Display only the list object's shallow size.
print("Note: this excludes the memory used by referenced objects such as the integers.")  # Clarify the limitation.

# 4. os

## Read environment variables safely

The `os` module provides operating-system interfaces, environment variables, directory operations, process information, and path-related functionality. Environment variables are commonly used for configuration such as API endpoints and database URLs.

Use `os.getenv(name, default)` to read an optional variable. Do not print secrets or commit them to Git. A `.env` file is not automatically secure: protect it with appropriate permissions and keep secret files out of version control.

In [ ]:
import os  # Import operating-system interfaces.

environment = os.getenv("APP_ENV", "development")  # Read an environment setting with a safe default.
api_key_present = bool(os.getenv("EXAMPLE_API_KEY"))  # Check whether a key exists without printing its value.
print("Application environment:", environment)  # Display non-sensitive configuration.
print("Example API key configured:", api_key_present)  # Report presence without exposing the secret.
print("Current working directory:", os.getcwd())  # Show the directory relative paths are resolved from.

## Create directories and inspect entries

`os.listdir()` lists names in a directory; `os.scandir()` can efficiently provide entry metadata and file-type checks. `os.makedirs(..., exist_ok=True)` creates a directory hierarchy without failing if it already exists.

Prefer `pathlib` for many path manipulations, but `os` remains useful for environment variables, permissions, process functions, and compatibility with APIs that expect OS-level values. This example uses a temporary directory so it does not clutter your project.

In [ ]:
import os  # Import directory and filesystem helpers.
import tempfile  # Create a disposable workspace.
from pathlib import Path  # Build filesystem paths safely.

with tempfile.TemporaryDirectory() as temp_dir:  # Automatically remove the temporary directory when done.
    base = Path(temp_dir)  # Convert the directory path to a Path object.
    os.makedirs(base / "reports" / "monthly", exist_ok=True)  # Create nested directories.
    (base / "reports" / "monthly" / "summary.txt").write_text("Report ready", encoding="utf-8")  # Create a test file.
    with os.scandir(base / "reports" / "monthly") as entries:  # Open a context-managed directory iterator.
        for entry in entries:  # Inspect each entry.
            print(entry.name, "file:", entry.is_file())  # Display name and file type.

## Walk a directory tree

`os.walk()` yields a directory, its subdirectories, and the files in it. This is useful for finding files recursively or building inventory reports. Be deliberate about recursion depth and exclusions because large folders can contain many files.

When scanning untrusted or shared directories, account for symbolic links, permissions, generated folders, and files that may change during traversal. Do not assume every entry can be opened successfully.

In [ ]:
import os  # Import directory-walking utilities.
import tempfile  # Use a disposable directory tree.
from pathlib import Path  # Manipulate test paths.

with tempfile.TemporaryDirectory() as temp_dir:  # Make a small example directory tree.
    root = Path(temp_dir)  # Turn the temporary path into a Path object.
    (root / "data").mkdir()  # Create a subdirectory.
    (root / "data" / "one.csv").write_text("id,value\n1,10\n", encoding="utf-8")  # Create a sample data file.
    (root / "readme.txt").write_text("Practice folder", encoding="utf-8")  # Create a text file.
    for current, directories, files in os.walk(root):  # Traverse each directory below the root.
        print("Folder:", current)  # Display the current folder.
        print("Subfolders:", directories)  # Display immediate subfolder names.
        print("Files:", files)  # Display immediate file names.

## Use OS-level file metadata and process information

The `os` module includes functions for file metadata and process information. `os.stat()` returns information such as size and modification time; `os.getpid()` returns the current process ID. Metadata availability and meaning can vary by operating system and filesystem.

Use `os.path.exists()` or, preferably in new code, `Path.exists()` to check existence—but remember the file can change between checking and using it. For robust file operations, handle the actual operation's exceptions as well.

In [ ]:
import os  # Import OS functions.
import tempfile  # Create a disposable file for metadata inspection.
from pathlib import Path  # Create and inspect the file path.

with tempfile.TemporaryDirectory() as temp_dir:  # Isolate the example from important files.
    path = Path(temp_dir) / "sample.txt"  # Define a test path.
    path.write_text("A small file", encoding="utf-8")  # Write example content.
    info = os.stat(path)  # Retrieve file metadata.
    print("Exists:", os.path.exists(path))  # Check the path currently exists.
    print("Size in bytes:", info.st_size)  # Display the file size.
    print("Current process ID:", os.getpid())  # Display this process's identifier.
    print("Modification timestamp:", info.st_mtime)  # Display the epoch timestamp of the last modification.

# 5. collections

## Count values with `Counter`

`collections.Counter` is a dictionary-like class designed for counting hashable objects. It supports counting items, finding the most common values, and combining counts. It is often more readable than manually updating a dictionary.

Use it for word counts, category counts, error frequencies, or repeated identifiers. For very large text processing, also consider memory use and whether tokenization should happen incrementally.

In [ ]:
from collections import Counter  # Import the specialized counting mapping.

regions = ["North", "West", "North", "East", "North", "West", "South"]  # Define example region values.
counts = Counter(regions)  # Count each distinct region.
print("Counts:", counts)  # Display the count mapping.
print("Most common:", counts.most_common(2))  # Display the two most frequent regions.
counts.update(["East", "East"])  # Add another batch of observations.
print("Updated counts:", counts)  # Display the changed counts.

## Group values with `defaultdict`

`defaultdict` creates a default value when a missing key is first accessed. For example, `defaultdict(list)` is convenient for grouping values, and `defaultdict(int)` is convenient for counts.

It can simplify aggregation loops. Be aware that merely accessing a missing key creates it, which differs from a normal dictionary's `.get()` method.

In [ ]:
from collections import defaultdict  # Import a mapping with automatic default values.

sales = [("North", 100), ("South", 80), ("North", 45), ("South", 25), ("West", 60)]  # Store region and amount pairs.
by_region = defaultdict(list)  # Each new region begins with an empty list.
for region, amount in sales:  # Process each sale.
    by_region[region].append(amount)  # Add the amount to that region's group.
totals = {region: sum(amounts) for region, amounts in by_region.items()}  # Convert groups into total sales.
print(dict(by_region))  # Display the grouped values as a regular dictionary.
print(totals)  # Display the regional totals.

## Use `deque` for queues

`collections.deque` is a double-ended queue. Appending and popping from either end are efficient, making it suitable for queues, sliding windows, and breadth-first search. Removing the first item from a normal list with `pop(0)` is slower for large lists because the remaining elements need to shift.

A deque can also have a `maxlen`, which discards older items when new items are appended beyond the limit. Use that behavior only when dropping old items is intended.

In [ ]:
from collections import deque  # Import the double-ended queue.

queue = deque(["job-A", "job-B"])  # Create a queue with two jobs.
queue.append("job-C")  # Add a job to the right-hand end.
next_job = queue.popleft()  # Process the oldest job from the left-hand end.
queue.appendleft("urgent-job")  # Add an urgent item to the front.
print("Processed:", next_job)  # Display the job removed from the queue.
print("Remaining:", list(queue))  # Convert to list for a readable display.
recent = deque(maxlen=3)  # Keep at most the three most recent items.
recent.extend([1, 2, 3, 4])  # Add four items to the bounded deque.
print("Recent values:", list(recent))  # Show that the oldest value was discarded.

## Use `ChainMap` and `namedtuple`

`ChainMap` searches multiple mappings in order without copying them into one new dictionary. It is useful for layered configuration, such as command-line values overriding defaults. Writing to a ChainMap normally updates the first mapping.

`namedtuple` creates tuple subclasses with named fields. It is lightweight and immutable, and can make records clearer than tuples indexed with `[0]` and `[1]`. For more complex mutable records, a dataclass may be a better fit.

In [ ]:
from collections import ChainMap, namedtuple  # Import layered mappings and named tuples.

defaults = {"theme": "light", "page_size": 20}  # Define default settings.
user_settings = {"theme": "dark"}  # Define settings that override defaults.
config = ChainMap(user_settings, defaults)  # Look up the user mapping before the defaults.
print(config["theme"], config["page_size"])  # Show the overridden theme and inherited page size.
config["page_size"] = 50  # Write to the first mapping, user_settings.
print(dict(config))  # Display the effective configuration.
Employee = namedtuple("Employee", ["name", "department", "salary"])  # Define a compact record type.
person = Employee("Mina", "Data", 75000)  # Create one named-tuple record.
print(person.name, person.salary)  # Access fields by meaningful names.

# 6. random

## Seed a reproducible random sequence

The `random` module generates pseudo-random values. Setting a seed makes a sequence reproducible, which is useful for simulations, tests, and sampling demonstrations. A seeded sequence is not truly unpredictable and must not be used for passwords, security tokens, or cryptographic keys.

Reproducibility requires recording other details too, including input data and library versions when numerical results matter.

In [ ]:
import random  # Import Python's pseudo-random generator.

random.seed(42)  # Initialize the generator to a known state.
first_run = [random.randint(1, 100) for _ in range(5)]  # Generate five reproducible integers.
random.seed(42)  # Reset the generator to the same starting state.
second_run = [random.randint(1, 100) for _ in range(5)]  # Generate the same sequence again.
print(first_run)  # Display the first sequence.
print("Same result:", first_run == second_run)  # Check that reseeding reproduced the result.

## Choose, sample, and shuffle

Use `choice` to choose one item, `sample` to select distinct elements without replacement, and `shuffle` to reorder a mutable sequence in place. These functions are useful for simulations and test data.

`sample` requires the sample size not to exceed the population size when sampling without replacement. `shuffle` returns `None`; it modifies the list itself. Do not confuse statistical sampling with security-sensitive random selection.

In [ ]:
import random  # Import random selection helpers.

random.seed(10)  # Make this example repeatable.
population = ["A", "B", "C", "D", "E"]  # Define a population.
one_item = random.choice(population)  # Select one element.
three_unique = random.sample(population, k=3)  # Select three distinct elements.
ordered = population.copy()  # Copy the original list so it stays unchanged.
random.shuffle(ordered)  # Shuffle the copy in place.
print("One item:", one_item)  # Display the selected item.
print("Sample without replacement:", three_unique)  # Display the sample.
print("Shuffled:", ordered)  # Display the reordered copy.

## Generate ranges and distributions

`randrange` chooses from a range, while `randint(a, b)` includes both endpoints. `uniform(a, b)` produces a floating-point sample from a continuous interval. `gauss(mu, sigma)` gives a sample from a normal distribution with the given mean and standard deviation.

These tools can simulate test data and uncertainty. Simulated data should be labeled as simulated and should not be presented as real observations.

In [ ]:
import random  # Import sampling functions.

random.seed(7)  # Set a reproducible seed.
integer_exclusive_stop = random.randrange(1, 10)  # Choose an integer from 1 through 9.
integer_inclusive_stop = random.randint(1, 10)  # Choose an integer from 1 through 10.
discount = random.uniform(0.05, 0.20)  # Simulate a floating-point discount between 5% and 20%.
measurement = random.gauss(100, 5)  # Simulate a measurement from a normal distribution.
print(integer_exclusive_stop, integer_inclusive_stop)  # Compare the integer range conventions.
print("Discount:", round(discount, 3))  # Display the simulated discount.
print("Measurement:", round(measurement, 2))  # Display the simulated measurement.

## Use `secrets` for security-sensitive randomness

The `random` module is for simulation, not cryptographic security. For password reset tokens, one-time codes, or other security-sensitive values, use the `secrets` module, which draws from the operating system's secure randomness source.

Never create a real password token using `random.randint()` or a predictable seed. Treat generated secrets as sensitive and avoid logging them.

In [ ]:
import secrets  # Import secure random-number helpers.

token = secrets.token_urlsafe(24)  # Generate a URL-safe token using secure randomness.
verification_code = secrets.randbelow(1_000_000)  # Generate an integer from 0 through 999999.
print("Token generated; length:", len(token))  # Report token metadata without printing the secret itself.
print("Verification-code length: 6 digits")  # Explain expected code formatting.
print("Example code formatted:", f"{verification_code:06d}")  # Display a demo code; do not log real verification codes in production.

# 7. re — Regular Expressions

## Use search, match, and fullmatch

The `re` module searches text using patterns. `re.search()` finds a match anywhere, `re.match()` checks from the beginning, and `re.fullmatch()` requires the whole string to match. Choosing the right operation makes validation behavior clearer.

Patterns such as `\d` match digits, `\w` matches word characters, `\s` matches whitespace, and `+` means one or more. Use raw strings like `r"\d+"` so Python does not interpret backslashes as string escapes.

In [ ]:
import re  # Import regular-expression support.

text = "Order #4832 completed"  # Define a string that contains an order number.
search_result = re.search(r"\d+", text)  # Find the first sequence of digits anywhere.
match_result = re.match(r"\d+", text)  # Attempt a match at the beginning of the text.
full_result = re.fullmatch(r"\d+", "4832")  # Require the complete supplied string to contain only digits.
print("Search:", search_result.group() if search_result else None)  # Display the found digit sequence.
print("Match at beginning:", bool(match_result))  # This is false because the text begins with letters.
print("Full numeric string:", bool(full_result))  # This is true because the whole string is digits.

## Capture groups and named groups

Parentheses create capture groups, allowing you to extract parts of a match. Named groups use `(?P<name>...)`, which can make parsed records easier to read. This is useful for logs, codes, filenames, and semi-structured text.

Regex is not always the best parser for nested formats such as arbitrary HTML or JSON; use a parser designed for those formats. Keep patterns readable and add tests for both expected and malformed input.

In [ ]:
import re  # Import the regex module.

line = "2026-10-26 ERROR Disk full"  # Define a predictable log line.
pattern = r"(?P<date>\d{4}-\d{2}-\d{2})\s+(?P<level>INFO|WARNING|ERROR)\s+(?P<message>.*)"  # Define named capture groups.
match = re.fullmatch(pattern, line)  # Require the entire line to match the expected format.
if match:  # Continue only when the line fits the format.
    record = match.groupdict()  # Convert named capture groups into a dictionary.
    print(record)  # Display the parsed fields.
else:  # Handle malformed lines.
    print("Unrecognized log format")  # Report the parsing failure.

## Substitute, split, and compile patterns

`re.sub()` replaces pattern matches, `re.split()` splits around matches, and `re.compile()` creates a reusable pattern. A compiled pattern is useful if it will be applied repeatedly.

Flags such as `re.IGNORECASE` change matching behavior. Be careful with `re.sub()` on real data: removing punctuation or whitespace can change meaning, identifiers, or legal names. Always define a cleaning rule based on the downstream task.

In [ ]:
import re  # Import regular-expression utilities.

text = "Data   Analyst; Python, SQL"  # Define text with repeated spaces and separators.
space_pattern = re.compile(r"\s+")  # Compile a pattern that matches one or more whitespace characters.
normalized = space_pattern.sub(" ", text).strip()  # Replace repeated whitespace with one space.
parts = re.split(r"[,;]\s*", normalized)  # Split on commas or semicolons and optional following spaces.
redacted = re.sub(r"\b\d{4}\b", "[YEAR]", "Report 2026 and code 1234")  # Replace four-digit sequences in a sample.
print(normalized)  # Display normalized text.
print(parts)  # Display the split values.
print(redacted)  # Display the substitution result.

## Avoid greedy matches and validate patterns

Quantifiers such as `*` and `+` are greedy by default: they match as much as they can while allowing the overall pattern to succeed. Adding `?` after a quantifier makes it lazy. Word boundaries `\b` help distinguish a word such as `cat` from a substring in `concatenate`.

Validation with regex usually checks format, not the full semantic meaning. For example, a pattern can match `2026-99-99` as a date-shaped string even though it is not a valid calendar date. Follow format validation with a date or domain-specific parser when necessary.

In [ ]:
import re  # Import regex support.
from datetime import datetime  # Use a calendar parser for semantic date validation.

html_fragment = "<b>first</b><b>second</b>"  # Create a simple string with repeated tags.
greedy = re.findall(r"<b>.*</b>", html_fragment)  # Greedy match spans from the first opening tag to the last closing tag.
lazy = re.findall(r"<b>.*?</b>", html_fragment)  # Lazy match returns each smallest matching tag segment.
date_text = "2026-02-28"  # Define a date-shaped string.
try:  # Attempt real calendar validation after regex format checking.
    parsed_date = datetime.strptime(date_text, "%Y-%m-%d")  # Parse year, month, and day.
except ValueError:  # Handle invalid calendar dates.
    print("Invalid date")  # Report invalid input.
else:  # Run only when parsing succeeds.
    print(parsed_date.date())  # Display the normalized date.
print("Greedy:", greedy)  # Show the longest match.
print("Lazy:", lazy)  # Show individual matches.

# 8. string

## Explore string constants and methods

The standard-library `string` module provides constants such as `ascii_letters`, `digits`, `punctuation`, and `whitespace`. These are useful for validation, simple parsing, and generating character sets.

String objects also have methods such as `.strip()`, `.split()`, `.join()`, `.replace()`, `.startswith()`, and `.casefold()`. Strings are immutable: the methods return new strings instead of modifying the original in place.

In [ ]:
import string  # Import reusable character constants.

print("ASCII letters:", string.ascii_letters)  # Show lowercase and uppercase ASCII letters.
print("Digits:", string.digits)  # Show decimal digits.
print("Punctuation sample:", string.punctuation[:10])  # Display the beginning of the punctuation constant.
raw = "  DATA Analysis  "  # Define a text value with surrounding whitespace.
cleaned = raw.strip().casefold()  # Remove surrounding whitespace and normalize case for comparisons.
print(cleaned)  # Display the cleaned value.
print("Original preserved:", repr(raw))  # Show that the original string was not changed.

## Use `string.Template` for simple templates

`string.Template` supports `$name` placeholders. It can be useful when users edit simple message templates. `substitute()` raises a `KeyError` when a required placeholder is missing; `safe_substitute()` leaves missing placeholders unresolved.

Do not use string templates or f-strings to assemble SQL from untrusted values. Use parameterized database queries for SQL. Template substitution is for text generation, not for escaping arbitrary programming languages.

In [ ]:
from string import Template  # Import a simple placeholder-based templating helper.

template = Template("Hello $name. Your report for $month is ready.")  # Define the placeholders in a message.
message = template.substitute(name="Mina", month="October")  # Replace all required placeholders.
print(message)  # Display the generated message.
partial = Template("Hello $name; owner: $owner").safe_substitute(name="Ali")  # Safely leave a missing placeholder unresolved.
print(partial)  # Display the partial substitution without raising KeyError.

## Translate characters with `maketrans` and `translate`

`str.maketrans()` creates a translation table, and `.translate()` applies it to a string. It can replace several individual characters efficiently or delete selected characters when they map to `None`.

This is character-level transformation, not full natural-language normalization. Do not delete punctuation or accents without checking the requirements of your data and language.

In [ ]:
text = "data-analysis, version 2.0!"  # Define text with characters to transform.
table = str.maketrans({"-": " ", ",": None, "!": None})  # Replace hyphens with spaces and delete selected punctuation.
cleaned = text.translate(table)  # Apply the translation table to the string.
print(cleaned)  # Display the transformed text.

## Format values and align columns

f-strings are the common modern way to insert expressions into strings. Format specifications can control decimal places, signs, alignment, and thousands separators. `string.Formatter` also exposes formatting behavior, though ordinary code usually needs only f-strings.

Use formatting for presentation rather than for the underlying calculation. Store numeric data as numeric types and apply formatting only when displaying or exporting results.

In [ ]:
import string  # Import the Formatter class for an introspection example.

revenue = 1234567.891  # Define a numeric revenue value.
margin = 0.2473  # Define a margin as a decimal fraction.
print(f"Revenue: ${revenue:,.2f}")  # Display currency with a thousands separator and two decimals.
print(f"Gross margin: {margin:.1%}")  # Display a decimal as a percentage.
print(f"{'Region':<12}{'Revenue':>14}")  # Align column headings left and right.
print(f"{'North':<12}{revenue:>14,.2f}")  # Align the formatted value under its heading.
print("Fields:", list(string.Formatter().parse("{name} earned {amount:.2f}")))  # Inspect a simple format string's components.

# 9. pathlib

## Build paths with `Path`

`pathlib.Path` provides an object-oriented way to work with filesystem paths. Use `/` to join path parts; Python selects the appropriate path representation for the current operating system. This is usually easier to read than manually concatenating strings.

A relative path is interpreted relative to the process's current working directory, not necessarily the notebook's location. When writing reusable code, make the expected base directory explicit and document how the program is launched.

In [ ]:
from pathlib import Path  # Import the Path class.

project = Path("my_project")  # Define a relative project folder.
data_file = project / "data" / "sales.csv"  # Join path components with the / operator.
print("Path:", data_file)  # Display the resulting path.
print("Absolute path:", data_file.resolve())  # Display the resolved path for diagnostics.
print("Name:", data_file.name)  # Display the filename.
print("Suffix:", data_file.suffix)  # Display the file extension.

## Read and write text files

`Path.read_text()` and `Path.write_text()` are concise methods for text files. Specify an encoding such as UTF-8 to avoid platform-dependent assumptions. For very large files, stream with `open()` rather than loading all content into memory.

Use temporary directories for exercises that create or modify files. Avoid overwriting important files just to test a method, and handle `OSError` when the operation may fail due to permissions or missing paths.

In [ ]:
from pathlib import Path  # Import path helpers.
from tempfile import TemporaryDirectory  # Create a safe temporary workspace.

with TemporaryDirectory() as temp_dir:  # Clean up all example files on exit.
    report_path = Path(temp_dir) / "reports" / "summary.txt"  # Define a nested path for the file.
    report_path.parent.mkdir(parents=True, exist_ok=True)  # Create parent directories before writing.
    report_path.write_text("Revenue increased.\nMargin was stable.\n", encoding="utf-8")  # Write UTF-8 text.
    content = report_path.read_text(encoding="utf-8")  # Read all text back into memory.
    print(content)  # Display the loaded text.
    print("Line count:", len(content.splitlines()))  # Count logical lines in the content.

## Find files with glob patterns

`Path.glob()` searches one directory level according to a pattern; `Path.rglob()` searches recursively. Patterns such as `*.csv` match file names. Use them to discover input files, create inventory reports, and process batches of data.

For large or uncontrolled directories, limit scope and exclude generated directories such as `.git`, virtual environments, caches, and large output folders. Do not assume the results are sorted unless you sort them yourself.

In [ ]:
from pathlib import Path  # Import filesystem path support.
from tempfile import TemporaryDirectory  # Create a temporary directory tree.

with TemporaryDirectory() as temp_dir:  # Keep all created examples temporary.
    root = Path(temp_dir)  # Represent the temporary root as a Path.
    (root / "data").mkdir()  # Create a data directory.
    (root / "data" / "a.csv").write_text("id\n1\n", encoding="utf-8")  # Create a CSV file.
    (root / "data" / "b.csv").write_text("id\n2\n", encoding="utf-8")  # Create another CSV.
    (root / "readme.md").write_text("# Demo", encoding="utf-8")  # Create a Markdown file.
    csv_files = sorted(root.rglob("*.csv"))  # Find CSV files recursively and sort the paths.
    print([path.name for path in csv_files])  # Display only file names.

## Inspect, rename, and validate paths

`Path.exists()`, `is_file()`, `is_dir()`, `stat()`, `suffix`, and `stem` help you inspect filesystem objects. `rename()` and `replace()` change path names; `replace()` overwrites an existing target where permitted, so use it carefully.

Checking existence and then acting can have a race condition if another process changes the path. Prefer performing the operation and handling its error. For untrusted path input, resolve the path and verify it remains inside the intended directory before accessing it.

In [ ]:
from pathlib import Path  # Import path operations.
from tempfile import TemporaryDirectory  # Isolate changes to a temporary folder.

with TemporaryDirectory() as temp_dir:  # Create a disposable folder.
    root = Path(temp_dir).resolve()  # Resolve the root directory.
    source = root / "draft.txt"  # Define a source filename.
    source.write_text("Draft report", encoding="utf-8")  # Create the source file.
    print(source.is_file(), source.stem, source.suffix)  # Inspect its type and name parts.
    destination = source.with_name("final.txt")  # Construct a sibling path with a new name.
    source.rename(destination)  # Rename the source file.
    print("Destination exists:", destination.exists())  # Confirm the renamed file exists.

# 10. shutil

## Copy files and preserve metadata

`shutil.copy()` copies file contents and permission bits, while `shutil.copy2()` attempts to preserve more metadata, such as modification times. Neither copies every possible filesystem attribute or security label on every platform.

Use explicit source and destination paths. Make sure the destination is correct before copying, particularly in scripts that process many files. The example copies within a temporary directory.

In [ ]:
import shutil  # Import high-level file operations.
from pathlib import Path  # Build path objects.
from tempfile import TemporaryDirectory  # Use a disposable directory.

with TemporaryDirectory() as temp_dir:  # Create a safe temporary workspace.
    root = Path(temp_dir)  # Convert the workspace to a Path.
    source = root / "source.txt"  # Define the source file.
    source.write_text("Practice copy operation", encoding="utf-8")  # Create the source contents.
    destination = root / "copy.txt"  # Define the copy destination.
    shutil.copy2(source, destination)  # Copy the file and preserve available metadata.
    print(destination.read_text(encoding="utf-8"))  # Check the copied content.
    print("Sizes match:", source.stat().st_size == destination.stat().st_size)  # Compare file sizes.

## Copy a directory tree

`shutil.copytree()` copies a directory and its contents. By default, the destination must not already exist; the `dirs_exist_ok=True` option can merge into an existing directory in modern Python versions. Use `ignore=` when some patterns should be excluded.

For repeatable pipelines, define exactly what should be copied and what should be skipped. Avoid copying virtual environments, huge caches, credentials, or private files into a repository or shared artifact by accident.

In [ ]:
import shutil  # Import directory copy functions.
from pathlib import Path  # Work with filesystem paths.
from tempfile import TemporaryDirectory  # Provide a temporary workspace.

with TemporaryDirectory() as temp_dir:  # Create example source and destination folders.
    root = Path(temp_dir)  # Define the workspace.
    source = root / "source"  # Set the source folder.
    (source / "data").mkdir(parents=True)  # Create a nested subfolder.
    (source / "data" / "file.csv").write_text("id,value\n1,10\n", encoding="utf-8")  # Create a nested file.
    destination = root / "backup"  # Set the destination folder.
    shutil.copytree(source, destination)  # Recursively copy the source folder.
    copied_file = destination / "data" / "file.csv"  # Build the copied file path.
    print(copied_file.exists(), copied_file.read_text(encoding="utf-8"))  # Verify the result.

## Move files and safely remove temporary trees

`shutil.move()` moves a file or directory and may copy then remove it if it crosses filesystems. `shutil.rmtree()` deletes an entire directory tree, so its target must be correct and preferably constrained to a temporary workspace.

Never test recursive deletion using a guessed path, user-provided path, or your project root. Use a `TemporaryDirectory` context manager in exercises; it cleans up the example automatically when the block exits.

In [ ]:
import shutil  # Import move and copy utilities.
from pathlib import Path  # Build paths.
from tempfile import TemporaryDirectory  # Keep destructive operations inside temporary storage.

with TemporaryDirectory() as temp_dir:  # Create a folder that Python will remove on exit.
    root = Path(temp_dir)  # Represent the root path.
    source = root / "old_name.txt"  # Define the original file.
    source.write_text("Move me", encoding="utf-8")  # Create the file.
    destination = root / "new_name.txt"  # Define the destination.
    shutil.move(str(source), str(destination))  # Move the file to its new path.
    print("Moved:", destination.exists(), "| Source remains:", source.exists())  # Verify the move.
    disposable = root / "disposable"  # Define a temporary folder to remove.
    disposable.mkdir()  # Create the folder.
    (disposable / "temp.txt").write_text("temporary", encoding="utf-8")  # Create a file inside it.
    shutil.rmtree(disposable)  # Remove only this known temporary subtree.
    print("Disposable folder remains:", disposable.exists())  # Verify removal.

## Create archives and inspect disk usage

`shutil.make_archive()` creates archives in formats such as ZIP or TAR. `shutil.disk_usage()` reports total, used, and free disk space for a path. These are useful for packaging datasets or monitoring storage.

Archive extraction requires special care when handling untrusted archives: filenames inside an archive can attempt path traversal, and archives can expand to very large sizes. Use trusted inputs or a safe extraction approach and enforce size limits.

In [ ]:
import shutil  # Import archive and disk utilities.
from pathlib import Path  # Work with filesystem paths.
from tempfile import TemporaryDirectory  # Isolate archive files.

with TemporaryDirectory() as temp_dir:  # Create temporary input and archive paths.
    root = Path(temp_dir)  # Define the working root.
    folder = root / "report_files"  # Create a folder to archive.
    folder.mkdir()  # Make the directory.
    (folder / "summary.txt").write_text("Synthetic report", encoding="utf-8")  # Add a sample file.
    archive_path = shutil.make_archive(str(root / "report_archive"), "zip", root_dir=folder)  # Create a ZIP archive from the folder contents.
    usage = shutil.disk_usage(root)  # Inspect the filesystem containing the temporary directory.
    print("Archive exists:", Path(archive_path).exists())  # Verify archive creation.
    print("Archive bytes:", Path(archive_path).stat().st_size)  # Display archive size.
    print("Free bytes:", usage.free)  # Display available disk space.

# 11. urllib

## Parse a URL into components

The `urllib.parse` module parses and builds URLs. `urlparse()` separates scheme, network location, path, query, and fragment. `parse_qs()` turns query parameters into lists of values, because a parameter name can appear more than once.

Parsing is not the same as validating that a URL is safe to fetch. For requests made from user-supplied URLs, check the allowed host, schemes, redirects, and network policy to avoid server-side request forgery or access to internal services.

In [ ]:
from urllib.parse import urlparse, parse_qs  # Import URL parsing helpers.

url = "https://example.com/search?q=data+analysis&page=2#results"  # Define a sample URL.
parsed = urlparse(url)  # Split the URL into its standard components.
query_values = parse_qs(parsed.query)  # Parse query parameters into lists of values.
print("Scheme:", parsed.scheme)  # Display https.
print("Host:", parsed.netloc)  # Display the network location.
print("Path:", parsed.path)  # Display the path.
print("Query parameters:", query_values)  # Display parsed query values.
print("Fragment:", parsed.fragment)  # Display the part after the # character.

## Build query strings and join relative URLs

Use `urlencode()` to encode query parameters, and `urljoin()` to resolve a relative link against a base URL. This avoids many errors caused by manual string concatenation.

Be aware that a leading slash in a relative link replaces the base path, and an absolute URL can replace the base host entirely. When extracting links from pages, validate each final destination before fetching it if the source is untrusted.

In [ ]:
from urllib.parse import urlencode, urljoin  # Import URL building helpers.

params = {"q": "data analyst", "page": 2, "tags": ["python", "sql"]}  # Define query parameters.
query_string = urlencode(params, doseq=True)  # Encode list values as repeated parameters.
url = f"https://example.com/search?{query_string}"  # Build a URL using the encoded query string.
base = "https://example.com/docs/guides/"  # Define the base URL.
relative = "../api"  # Define a relative link.
joined = urljoin(base, relative)  # Resolve the relative link against the base URL.
print(url)  # Display the encoded search URL.
print(joined)  # Display the resolved relative URL.

## Create an HTTP request object

`urllib.request.Request` represents a request, including its URL, data, and headers. It can be used with `urlopen()` for standard-library HTTP calls. `urllib.error` provides exceptions such as `HTTPError` and `URLError`.

A real network call is intentionally not performed here. In an application, use a timeout, handle HTTP and network errors, inspect the returned content type, and limit how much data you read. Do not fetch sites or endpoints you are not authorized to access.

In [ ]:
from urllib.request import Request  # Import the HTTP request object.

request = Request(  # Create a request without sending it.
    "https://example.com/api/items?page=1",  # Supply the URL.
    headers={"User-Agent": "LearningPython/1.0", "Accept": "application/json"},  # Add headers expected by the server.
    method="GET",  # Explicitly name the HTTP method.
)  # Finish the request definition.
print("URL:", request.full_url)  # Display the target URL.
print("Method:", request.get_method())  # Display the method.
print("User-Agent:", request.get_header("User-agent"))  # Inspect the supplied User-Agent value.

## Understand URL errors and safe fetching

If using `urlopen()`, pass a timeout and catch expected errors such as `urllib.error.HTTPError` and `urllib.error.URLError`. `HTTPError` is also a file-like response, and its status code can provide useful diagnostic information. `URLError` often indicates a connection or URL-related problem.

For fetching user-provided URLs, validate schemes and hostnames, restrict redirects if necessary, and avoid private IP ranges and internal services. A URL parser alone is not a security boundary. This example defines a small wrapper but does not call the network.

In [ ]:
from urllib.request import Request, urlopen  # Import request and URL opening helpers.
from urllib.error import HTTPError, URLError  # Import common network exceptions.

def fetch_text(url, timeout=5):  # Define a small bounded text-fetching helper.
    request = Request(url, headers={"User-Agent": "LearningPython/1.0"})  # Supply a descriptive client identifier.
    try:  # Handle known HTTP and URL-level errors.
        with urlopen(request, timeout=timeout) as response:  # Open the response with a finite timeout.
            content_type = response.headers.get("Content-Type", "")  # Inspect the response type.
            body = response.read(100_000)  # Read at most 100 KB for this example.
            return content_type, body.decode("utf-8", errors="replace")  # Return a decoded sample.
    except HTTPError as error:  # Handle an HTTP error status from the server.
        raise RuntimeError(f"HTTP status error: {error.code}") from error  # Add context while preserving the cause.
    except URLError as error:  # Handle a URL or network problem.
        raise RuntimeError(f"Network error: {error.reason}") from error  # Raise a contextual application error.

print("Fetcher defined; call it only with a permitted, validated URL.")  # Avoid a live request during notebook execution.

# 12. asyncio

## Understand coroutines and `asyncio.run`

`asyncio` supports concurrent I/O work using `async` and `await`. An `async def` function creates a coroutine object; it does not run to completion until it is awaited or scheduled. `asyncio.run()` starts an event loop for a top-level async program and should not be called inside an already-running event loop, such as many notebook environments.

In Jupyter, an event loop may already be running. The notebook examples use `await` directly when appropriate. In a regular `.py` script, the usual entry point is `asyncio.run(main())`.

In [ ]:
import asyncio  # Import Python's asynchronous I/O tools.

async def greet():  # Define a coroutine function.
    await asyncio.sleep(0)  # Yield control to the event loop without blocking the thread.
    return "Hello from async Python"  # Return a value to the caller.

message = await greet()  # In Jupyter, await the coroutine directly.
print(message)  # Display the returned result.
# In a standalone script, a common pattern is: asyncio.run(greet()).  # Use this pattern outside an active event loop.

## Run tasks concurrently with `gather`

`asyncio.gather()` schedules several awaitables and returns their results in the same order as the inputs. It is useful when tasks spend time waiting for network or other asynchronous I/O.

Concurrency does not make CPU-bound Python code faster by itself. Use async primarily for operations whose libraries support asynchronous waiting. If one coroutine raises, `gather()` normally propagates the exception; other tasks may continue unless cancellation or structured concurrency is handled deliberately.

In [ ]:
import asyncio  # Import the async framework.

async def simulated_fetch(name, delay):  # Model an asynchronous I/O task without the network.
    print("Starting", name)  # Mark the task's start.
    await asyncio.sleep(delay)  # Yield control while waiting.
    print("Finished", name)  # Mark the task's completion.
    return {"name": name, "delay": delay}  # Return a result.

results = await asyncio.gather(  # Run all three coroutines concurrently.
    simulated_fetch("endpoint-A", 0.03),  # Schedule the first task.
    simulated_fetch("endpoint-B", 0.01),  # Schedule the second task.
    simulated_fetch("endpoint-C", 0.02),  # Schedule the third task.
)  # Wait for all results.
print("Results in input order:", results)  # Results retain the order of the awaitables supplied to gather.

## Create tasks and impose a timeout

`asyncio.create_task()` schedules a coroutine to run concurrently and returns a Task object. You can await the task later, inspect its result, or cancel it. A task should not be left unobserved if it may raise an exception.

`asyncio.wait_for()` imposes a deadline on an awaitable. When the timeout expires, it cancels the awaited operation and raises `asyncio.TimeoutError` (an alias of built-in `TimeoutError` in current Python versions). Cancellation is cooperative, so code should clean up resources properly.

In [ ]:
import asyncio  # Import asynchronous task helpers.

async def slow_operation():  # Define work that takes longer than our deadline.
    await asyncio.sleep(0.2)  # Simulate I/O waiting.
    return "completed"  # Return a value if not cancelled.

task = asyncio.create_task(slow_operation())  # Schedule the coroutine as a task.
try:  # Guard a wait that may time out.
    result = await asyncio.wait_for(task, timeout=0.01)  # Impose a short deadline.
except TimeoutError:  # Handle the deadline expiration.
    print("Operation timed out")  # Explain the outcome.
    if not task.done():  # Check whether the task is still pending.
        task.cancel()  # Request cancellation.
    await asyncio.gather(task, return_exceptions=True)  # Retrieve cancellation to avoid an unobserved task.
else:  # Run when the work completes in time.
    print(result)  # Display the result.

## Limit concurrency with a semaphore

Launching thousands of concurrent tasks at once can overload a remote service or exhaust local resources. `asyncio.Semaphore` limits how many tasks may enter a protected section concurrently.

Use a semaphore together with timeouts, respectful rate limits, error handling, and cancellation. The example simulates a limited-capacity service; it does not send requests to a real website.

In [ ]:
import asyncio  # Import asynchronous coordination tools.

async def run_limited(name, semaphore, active_state):  # Define one simulated unit of work.
    async with semaphore:  # Enter only when the semaphore has capacity.
        active_state["active"] += 1  # Count how many tasks are currently inside.
        active_state["maximum"] = max(active_state["maximum"], active_state["active"])  # Track observed concurrent work.
        await asyncio.sleep(0.02)  # Simulate waiting for I/O.
        active_state["active"] -= 1  # Record that this task has left the protected section.
        return f"{name} done"  # Return a short result.

limit = asyncio.Semaphore(3)  # Permit at most three concurrent tasks in the critical section.
state = {"active": 0, "maximum": 0}  # Track concurrent work for this demonstration.
results = await asyncio.gather(*(run_limited(f"task-{i}", limit, state) for i in range(10)))  # Run ten tasks with a concurrency cap.
print(results)  # Display results after all tasks finish.
print("Maximum active at once:", state["maximum"])  # Confirm concurrency never exceeded the cap.